# Phase 2: the word recogniser

Trains and checks the recogniser of whole handwritten Meitei Mayek words on the synthetic
words of Phase 1, and the baseline it has to beat:

1. **The first paper's weights** (Hugging Face): ConvNeXt-T's weights on TUMMHCD start the
   recogniser's encoder, and the whole ensemble is the baseline's classifier.
2. **The character language model** (n-grams of the training words), order and weighting
   chosen on the validation words. Writes `results/phase2_lm.json`.
3. **A look at the training data**: speed of rendering and of the GPU, and a batch as the
   network sees it (augmented).
4. **Training**, one cell per run (about 2.5 hours each on an A100 40 GB, 10 hours for the
   four): the second round, with 10% of the training words scrambled, is ConvNeXt-T started
   from TUMMHCD (two seeds), the same started from ImageNet, and a small CNN from scratch.
   A run goes on where it stopped when its cell is run again. Writes
   `results/phase2_train_<run>.json`.
5. **Validation**: greedy decoding, and beam search with the language model (its weight
   and the bonus per character chosen here). Writes `results/phase2_val_<run>.json`
   and a sheet of misread words per run.
6. **The baseline on validation**: words cut into characters by the synthesiser's own
   boxes (perfect segmentation), each character classified by the first paper's ensemble,
   with zones and the language model. Writes `results/phase2_baseline_val.json`.
7. **Test, once**, only after every choice is made on validation: set `RUN_TEST = True` in
   the first cell, then run the first cell, the setup cell and the section 7 cell (nothing
   else). Writes `results/phase2_test_<run>.json` and `results/phase2_baseline_test.json`.

Needs an A100 (Runtime > Change runtime type). Inputs on Drive, from the Phase 1 notebook:
`WORK/glyphs/*.npz`, `WORK/lexicon/*.tsv` and `WORK/synth/{val,test}.tar`. Checkpoints go
to `WORK/runs`. Afterwards, send back (or commit) the files in `WORK/results`, and the
error sheets `WORK/runs/*/val_errors.png`.

Optional secrets (Colab key icon): `HF_TOKEN`, only if the weights cannot be downloaded
without one; `GITHUB_TOKEN`, only while the repository is private.

In [ ]:
# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-recognition"
REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
BRANCHES = ["claude/intelligent-euler-rx42yl", "main"]  # the first that has the Phase 2 code is used
# the first project's package (the baseline's classifier), pinned as in Phase 1
MAYEK = ("git+https://github.com/chingkheinganba231005/Handwritten-Meitei-Mayek-Recognition"
         "@0d2c6e5b4c6589c635bf18048755137c1f3b519d")
HF_MODEL = "Chingkheinganba/handwritten-meitei-mayek-recognition"   # the first paper's weights
# the first project's runs folder: its "dev" networks, trained without our validation part, check the
# baseline on the validation set (the released networks were trained on the validation part too)
FIRST_RUNS = "/content/drive/MyDrive/tummhcd98/runs"
STEPS = 60000       # training steps per run, 64 words each
# name: settings of mayek_htr.train.TrainConfig. Second round (owner, 26 September 2026): 10% of the
# training words scrambled (random letters of each kind), the main model with two seeds, ImageNet and
# the small CNN once. The first round's runs (convnext_tummhcd, convnext_imagenet, crnn_scratch, without
# scrambled words) keep their folders and results.
RUNS = {
    "round2_convnext_tummhcd": {"encoder": "convnext_tiny", "init": "tummhcd", "scrambled": 0.1},
    "round2_convnext_tummhcd_seed1": {"encoder": "convnext_tiny", "init": "tummhcd", "scrambled": 0.1,
                                      "seed": 1, "data_seed": 1001},
    "round2_convnext_imagenet": {"encoder": "convnext_tiny", "init": "imagenet", "scrambled": 0.1},
    "round2_crnn_scratch": {"encoder": "small_cnn", "init": "none", "scrambled": 0.1},
}
RUN_TEST = False    # True only when every choice is made on validation: the test set is used once

In [ ]:
import glob, json, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")  # only needed while the repository is private
except Exception:
    token = None

def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token or "\0", "***")
    if out.strip():
        print(out)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")

def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")

REPO_DIR = "/content/repo"
if not os.path.exists(f"{REPO_DIR}/mayek_htr/train.py"):
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    for branch in BRANCHES:
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch, url, REPO_DIR],
                           capture_output=True, text=True)
        if r.returncode == 0 and os.path.exists(f"{REPO_DIR}/mayek_htr/train.py"):
            break
    else:
        raise SystemExit("no branch in BRANCHES has the Phase 2 code. git: "
                         + r.stderr.replace(token or "\0", "***"))
    run("git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git")
else:  # a runtime left from an earlier session: bring the code up to date (its outputs are on Drive)
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    branch = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    run("git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", url, branch)
    run("git", "-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
run(sys.executable, "-m", "pip", "install", "-q", MAYEK, "timm", "huggingface_hub", "scipy")
for d in ("results", "runs", "lm", "baseline"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
for f in ("glyphs/train.npz", "glyphs/val.npz", "glyphs/test.npz", "lexicon/train.tsv", "lexicon/val.tsv",
          "lexicon/test.tsv", "synth/val.tar", "synth/test.tar"):
    if not os.path.exists(f"{WORK}/{f}"):
        raise SystemExit(f"{WORK}/{f} is missing: run the Phase 1 notebook first")
    os.makedirs(os.path.dirname(f"/content/{f}"), exist_ok=True)
    if not os.path.exists(f"/content/{f}"):
        shutil.copy(f"{WORK}/{f}", f"/content/{f}")   # local copies read faster than Drive
WEIGHTS = "/content/weights"                  # the first paper's weights (section 1)
if not os.path.exists(WEIGHTS) and os.path.exists(f"{WORK}/weights"):
    shutil.copytree(f"{WORK}/weights", WEIGHTS)  # downloaded in an earlier session
LM = f"{WORK}/lm/char_lm.pkl"                  # the character language model (section 2)
WORKERS = max((os.cpu_count() or 4) - 2, 2)    # processes rendering training words

def train_run(name):
    """Trains run `name` of RUNS, or goes on with it (section 4)."""
    flags = [a for k, v in RUNS[name].items() for a in ("--" + k.replace("_", "-"), v)]
    stream(sys.executable, "-u", "scripts/train_recogniser.py", "--glyphs", "/content/glyphs/train.npz",
           "--lexicon", "/content/lexicon/train.tsv", "--sizes", "results/glyph_sizes_tummhcd.json",
           "--val", "/content/synth/val.tar", "--run-dir", f"{WORK}/runs/{name}", "--tummhcd-dir", WEIGHTS,
           "--results", f"results/phase2_train_{name}.json", "--steps", STEPS, "--workers", WORKERS, *flags)
    shutil.copy(f"results/phase2_train_{name}.json", f"{WORK}/results/")

run("git", "log", "-1", "--format=%h %s")
run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")
print("CPUs:", os.cpu_count())

## 1. The first paper's weights

Downloaded once and kept on Drive (`WORK/weights`); later sessions copy them from there.

In [ ]:
if not glob.glob(f"{WEIGHTS}/**/config.json", recursive=True):
    from huggingface_hub import snapshot_download
    try:
        hf_token = userdata.get("HF_TOKEN")  # only if the download needs one
    except Exception:
        hf_token = None
    snapshot_download(HF_MODEL, local_dir=WEIGHTS, token=hf_token)
    shutil.copytree(WEIGHTS, f"{WORK}/weights", dirs_exist_ok=True)
from mayek_htr.model import find_model_dir
MODEL_DIR = find_model_dir(WEIGHTS)
members = json.load(open(MODEL_DIR / "config.json"))["members"]
print(MODEL_DIR, [m["name"] for m in members])

## 2. The character language model

Kneser-Ney n-grams over the training words (and numbers and full stops, as in the
synthetic words), for orders 3 to 7 and three weightings of the words; the one with the
lowest perplexity on the validation words is kept.

In [ ]:
run(sys.executable, "scripts/build_char_lm.py", "--train", "/content/lexicon/train.tsv",
    "--val", "/content/lexicon/val.tsv", "--out", LM, "--results", "results/phase2_lm.json")
shutil.copy("results/phase2_lm.json", f"{WORK}/results/")

## 3. A look at the training data

Training words are rendered while training, by the CPU's worker processes. If rendering
cannot keep up with the GPU, the training log says how much of the time it waits for data.
Below: words per second rendered by one process, GPU steps per second for each encoder,
and the hours a run should take. Then a batch before and after augmentation.

In [ ]:
import numpy as np, torch
from IPython.display import Image as Show, display
from PIL import Image
from mayek_words.glyphs import GlyphStore
from mayek_words.lexicon import Lexicon
from mayek_words.synth import Words, WordSynth, load_priors
from mayek_htr import images
from mayek_htr.augment import AUG, augment
from mayek_htr.data import SynthStream
from mayek_htr.model import MEAN, STD, build

store = GlyphStore.load("/content/glyphs/train.npz")
words = Words(WordSynth(store, load_priors(sizes="results/glyph_sizes_tummhcd.json")),
              Lexicon.load("/content/lexicon/train.tsv"), seed=99, scrambled=0.1)
t0 = time.time()
widths = [images.normalise(words[i].image).shape[1] for i in range(300)]
per_word = (time.time() - t0) / 300
print(f"rendering: {1000 * per_word:.1f} ms per word per process; with {WORKERS} workers about "
      f"{WORKERS / per_word:.0f} words/s; mean width {np.mean(widths):.0f} px at 64 px high")

for enc in ("convnext_tiny", "small_cnn"):
    model = build(enc, init="none").cuda().train()
    opt = torch.optim.AdamW(model.parameters(), 1e-4)
    x = torch.rand(64, 1, 64, 192, device="cuda")
    w = torch.full((64,), 192, device="cuda")
    for k in range(25):
        if k == 5:
            torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits, _ = model(x, w)
        loss = logits.float().logsumexp(-1).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    step = (time.time() - t0) / 20
    need = 64 / (WORKERS / per_word)   # seconds of rendering per step
    print(f"{enc}: {1000 * step:.0f} ms per step on the GPU; a run of {STEPS} steps takes about "
          f"{STEPS * max(step, need) / 3600:.1f} h ({'data' if need > step else 'GPU'} bound)")
    del model, opt

batch = next(iter(SynthStream(words, batch_size=16, pool=1)))
x = batch["x"].cuda().float() / 255
torch.manual_seed(0)
y = augment(x, batch["widths"].cuda(), AUG)
rows = [np.concatenate([a, np.full((64, 8), 128, np.uint8), b], 1) for a, b in
        zip((255 - x[:, 0] * 255).byte().cpu().numpy(), (255 - y[:, 0] * 255).byte().cpu().numpy())]
sheet = np.concatenate([np.pad(r, ((4, 4), (0, 0)), constant_values=255) for r in rows], 0)
Image.fromarray(sheet).save("/content/batch.png")
display(Show("/content/batch.png"))
print(batch["texts"])

## 4. Training

One cell per run; run them one after another (the main model first). Nothing changes
between runs: each run's settings are in `RUNS` (first cell), and sections 1 to 3 are done
once for all of them. In every run a tenth of the training words are scrambled: a lexicon
word with each letter, lonsum letter and vowel sign replaced by a random one of its kind, so
that every letter is seen in every context (the first round read ꯘ as ꯗ where ꯗ is common). A
run saves its state every 1,000 steps to `WORK/runs/<run>`; if Colab disconnects, run the
first two cells and the run's cell again and it goes on from there (a finished run's cell
only finishes again). Validation (greedy, 5,000 synthetic words) every 2,000 steps;
`best.pt` keeps the weights with the lowest character error rate.

In [ ]:
train_run("round2_convnext_tummhcd")

In [ ]:
train_run("round2_convnext_tummhcd_seed1")

In [ ]:
train_run("round2_convnext_imagenet")

In [ ]:
train_run("round2_crnn_scratch")

## 5. Validation

For each finished run: greedy decoding, and beam search with the language model, whose
weight (alpha) and bonus per character (beta) are chosen here on the validation words. The
sheet shows the first 48 misread words (reference above, reading below).

In [ ]:
import pandas as pd
from IPython.display import Image as Show, display

rows = []
for name in RUNS:
    ck = f"{WORK}/runs/{name}/best.pt"
    if not os.path.exists(ck):
        print(f"{name}: not trained yet")
        continue
    run(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", ck, "--set", "/content/synth/val.tar",
        "--lm", LM, "--tune", "--out", f"results/phase2_val_{name}.json",
        "--predictions", f"{WORK}/runs/{name}/val_predictions.tsv", "--sheet", f"{WORK}/runs/{name}/val_errors.png")
    shutil.copy(f"results/phase2_val_{name}.json", f"{WORK}/results/")
    r = json.load(open(f"results/phase2_val_{name}.json"))
    row = {"run": name, "step": r["step"], "CER": r["greedy"]["cer"], "WER": r["greedy"]["wer"],
           "CER LM": r["with_lm"]["cer"], "WER LM": r["with_lm"]["wer"],
           "alpha": r["lm"]["alpha"], "beta": r["lm"]["beta"]}
    row.update({f"{p} LM": v["accuracy"] for p, v in r["with_lm"]["pairs"].items()})
    rows.append(row)
display(pd.DataFrame(rows))
for name in RUNS:
    if os.path.exists(f"{WORK}/runs/{name}/val_errors.png"):
        print(name)
        display(Show(f"{WORK}/runs/{name}/val_errors.png"))

## 6. The baseline on validation

The validation words are rendered again to recover each character's box, then every
character is classified by the first paper's ensemble, both as its original TUMMHCD image
(isolated) and as cut from the word image (cut); with and without zones and the language
model. The released networks were trained on our validation part too, so here the
ensemble is the first paper's development networks (`FIRST_RUNS/<network>/dev/final.pt`,
trained without it), put together by `scripts/dev_ensemble.py`. Isolated, they should now
miss about 2% of the characters, as on TUMMHCD test, not almost none. The test (section 7)
uses the released networks, with the language model's weights chosen here.

In [ ]:
import pandas as pd
from IPython.display import display

DEV_WEIGHTS = "/content/weights_dev/first_paper_dev"
if os.path.isdir(FIRST_RUNS):
    if not os.path.exists(f"{DEV_WEIGHTS}/config.json"):
        run(sys.executable, "scripts/dev_ensemble.py", "--release", WEIGHTS, "--runs", FIRST_RUNS, "--out", DEV_WEIGHTS)
    baseline_weights = DEV_WEIGHTS
else:
    print(f"{FIRST_RUNS} not found: the baseline is checked with the released networks, which have seen "
          "the validation characters, so its validation scores are too good")
    baseline_weights = WEIGHTS
run(sys.executable, "scripts/oracle_baseline.py", "--model-dir", baseline_weights, "--set", "/content/synth/val.tar",
    "--glyphs", "/content/glyphs/val.npz", "--lexicon", "/content/lexicon/val.tsv",
    "--sizes", "results/glyph_sizes_tummhcd.json", "--lm", LM, "--tune",
    "--out", "results/phase2_baseline_val.json", "--predictions", f"{WORK}/baseline/val_predictions.tsv")
shutil.copy("results/phase2_baseline_val.json", f"{WORK}/results/")
b = json.load(open("results/phase2_baseline_val.json"))
display(pd.DataFrame([{"input": form, "decoding": how, "CER": s["cer"], "WER": s["wer"]}
                      for form, v in b["variants"].items() for how, s in v.items()]))

## 7. Test, once

The 5,000 synthetic test words: words and character images that no run and no choice has
seen. Run this only after every choice (runs, checkpoints, language model, alpha and beta)
is made on validation, and only once: a result looked at and then acted on is no longer a
test. The only change in the notebook: `RUN_TEST = True` in the first cell. Then run the
first cell, the setup cell and this cell; no training and no sections 1 to 6. The choices
made on validation are read from `WORK/results`.

In [ ]:
if not RUN_TEST:
    print("RUN_TEST is False: the test set stays untouched")
else:
    chosen = {name: f"{WORK}/results/phase2_val_{name}.json" for name in RUNS}   # alpha and beta per run
    chosen["baseline"] = f"{WORK}/results/phase2_baseline_val.json"
    missing = [f for f in chosen.values() if not os.path.exists(f)]
    missing += [f"{WORK}/runs/{n}/best.pt" for n in RUNS if not os.path.exists(f"{WORK}/runs/{n}/best.pt")]
    if missing:
        raise SystemExit(f"train every run and run sections 5 and 6 first; missing: {missing}")
    for name in RUNS:
        run(sys.executable, "scripts/eval_recogniser.py", "--checkpoint", f"{WORK}/runs/{name}/best.pt",
            "--set", "/content/synth/test.tar", "--lm", LM, "--tuned", chosen[name],
            "--out", f"results/phase2_test_{name}.json", "--predictions", f"{WORK}/runs/{name}/test_predictions.tsv")
        shutil.copy(f"results/phase2_test_{name}.json", f"{WORK}/results/")
    run(sys.executable, "scripts/oracle_baseline.py", "--model-dir", WEIGHTS, "--set", "/content/synth/test.tar",
        "--glyphs", "/content/glyphs/test.npz", "--lexicon", "/content/lexicon/test.tsv",
        "--sizes", "results/glyph_sizes_tummhcd.json", "--lm", LM, "--tuned", chosen["baseline"],
        "--out", "results/phase2_baseline_test.json")
    shutil.copy("results/phase2_baseline_test.json", f"{WORK}/results/")

In [ ]:
print("Send back or commit:")
for f in sorted(glob.glob(f"{WORK}/results/phase2_*.json")):
    print(" ", f)
print("  and", f"{WORK}/runs/*/val_errors.png")